# Chain-of-Thought Knowledge Distillation from LLMs to SLMs with Reasoning-Quality Evaluation

In [4]:
# clone repo
# !git clone https://github.com/rihembenabdallah18/COT_lab.git
!git pull
%cd COT_lab

remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 10 (delta 3), reused 9 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 14.72 KiB | 3.68 MiB/s, done.
From https://github.com/rihembenabdallah18/COT_lab
   0f512a3..189257a  main       -> origin/main
   b36bd55..be18ddf  dev        -> origin/dev
Updating 0f512a3..189257a
Fast-forward
 notebooks/cot-gpt.ipynb   | 999 ++++++++++++++++++++++++++++++++++++++++++++++
 scripts/04_inference.sh   |  18 +
 src/inference/generate.py | 154 +++++++
 3 files changed, 1171 insertions(+)
 create mode 100644 notebooks/cot-gpt.ipynb
 create mode 100755 scripts/04_inference.sh
 create mode 100644 src/inference/generate.py
[Errno 2] No such file or directory: 'COT_lab'
/kaggle/working/COT_lab


In [ ]:
# install deps
!pip install torch==2.3.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install -r requirements.txt -q
!python -m spacy download en_core_web_sm -q

In [2]:
# check gpu set up 
import torch
print(torch.cuda.is_available())       # should print True
print(torch.cuda.get_device_name(0))   # should print Tesla T4

True
Tesla T4


## pipeline

In [4]:
!bash scripts/01_download.sh

[gsm8k] already downloaded at /kaggle/working/COT_lab/data/raw/gsm8k
[ho] already extracted at /kaggle/working/COT_lab/data/raw/ho_et_al_cots/gsm8k_zs_cot_text-davinci-002.json
HO ET AL. SCHEMA REPORT
[meta] {
  "dataset_key": "gsm8k",
  "base_model": "text-davinci-002",
  "completion_key": "zs_cot",
  "finetune_key": null,
  "train_key": null,
  "method": "ft_cot",
  "prediction_template": null,
  "parameters": {
    "temperature": 0,
    "max_tokens": 128,
    "reasoning_temperature": 0,
    "reasoning_max_tokens": 128
  },
  "epoch": null
}
[counts] teacher records: 8792 (GSM8K train=7473, test=1319, sum=8792)
[note] sample_index ranges over train+test concatenated; train=[0..7472], test=[7473..8791]
[fields] ['sample_index', 'completion_index', 'question', 'answer', 'reasoning_prompt', 'reasoning_completion', 'reasoning_finish_reason', 'prompt', 'completion']
[interpretation]
  reasoning_prompt    -> Q + 'Let's think step by step.' (input to step 1)
  reasoning_completion-> teacher

In [5]:
!bash scripts/02_filter.sh

GSM8K train rows: 7473
Set A (no filter): 7473 -> /kaggle/working/COT_lab/data/processed/set_a_nofilter.jsonl
Set B (Magister) : 3389 -> /kaggle/working/COT_lab/data/processed/set_b_magister.jsonl
  skipped: no_teacher=0 unparseable_gold=0 unparseable_teacher_pred=78 (these are still in A)
Set B keep rate (of A): 45.3%

--- 3 example records, Set A ---
{
  "sample_index": 0,
  "question": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natali...",
  "cot": "In April, Natalia sold 48 clips. \nIn May, she sold half as many clips. Half of 48 is 24, so she sold 24 clips in May. \n\n...",
  "gold_answer": "72",
  "teacher_predicted_answer": "72."
}
{
  "sample_index": 1,
  "question": "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?",
  "cot": "First, we need to figure out how many hours she worked. We know that she worked for 50 minutes, and that there are 60

In [10]:
!pip uninstall peft -y

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [11]:
!bash scripts/03_train_set_b.sh

2026-05-06 02:20:53.506201: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778034053.528058     338 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778034053.534947     338 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778034053.553250     338 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778034053.553283     338 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778034053.553286     338 computation_placer.cc:177] computation placer alr

In [13]:
!bash scripts/03_train_set_a.sh

2026-05-06 02:40:57.803437: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778035257.825717    6078 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778035257.832576    6078 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778035257.850475    6078 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778035257.850528    6078 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778035257.850536    6078 computation_placer.cc:177] computation placer alr

In [5]:
!bash scripts/04_inference.sh

=== baseline ===
[baseline] loading model from google/flan-t5-base
config.json: 1.40kB [00:00, 4.21MB/s]
tokenizer_config.json: 2.54kB [00:00, 7.31MB/s]
spiece.model: 100%|██████████████████████████| 792k/792k [00:00<00:00, 2.02MB/s]
tokenizer.json: 2.42MB [00:00, 49.5MB/s]
special_tokens_map.json: 2.20kB [00:00, 10.1MB/s]
model.safetensors:  54%|███████████▉          | 536M/990M [00:04<00:03, 116MB/s]
Loading weights: 100%|█| 282/282 [00:00<00:00, 1504.91it/s, Materializing param=
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
generation_config.json: 100%|███████████████████| 147/147 [00:00<00:00, 662kB/s]
[baseline] device=cuda, resuming from record 0
baseline: 100%|███████████████████████████████| 165/165 [14:38<00:00,  5.33s/it]
[baseline] done: 1319 examples in 879s (0.7s/ex) ->